# 04 — EMIT Spectral Extraction

Extract one EMIT spectrum for each sample point and save a point-by-wavelength table.

The wavelength list is read from the ENVI `.hdr` file, following the existing project workflow.  
Water-absorption regions used in the existing analysis are masked:

- 1340–1445 nm
- 1800–1955 nm

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
SPECTRAL_DIR = OUTPUT_DIR / "spectral_profiles"
SPECTRAL_DIR.mkdir(parents=True, exist_ok=True)

EMIT_DAT = DATA_DIR / "Emit.dat"
EMIT_HDR = DATA_DIR / "Emit.hdr"
SAMPLE_SHP = OUTPUT_DIR / "NDVI_Selected_Points.shp"
OUT_CSV = SPECTRAL_DIR / "EMIT_extracted_spectra.csv"

CLASSES = ["Stress", "Moderate", "Healthy"]

In [ ]:
def parse_wavelengths(hdr_path):
    text = hdr_path.read_text(encoding="utf-8", errors="ignore")
    match = re.search(r"wavelength\s*=\s*\{([^}]*)\}", text, flags=re.I | re.S)
    if not match:
        raise ValueError("Could not find wavelength list in the ENVI header.")
    return np.array(
        [float(x.strip()) for x in match.group(1).split(",") if x.strip()],
        dtype=float,
    )

wavelengths = parse_wavelengths(EMIT_HDR)

valid_band_mask = ~(
    ((wavelengths >= 1340) & (wavelengths <= 1445))
    | ((wavelengths >= 1800) & (wavelengths <= 1955))
)

clean_wavelengths = wavelengths[valid_band_mask]

print("Total bands:", len(wavelengths))
print("Bands after masking:", len(clean_wavelengths))

In [ ]:
gdf = gpd.read_file(SAMPLE_SHP)

with rasterio.open(EMIT_DAT) as src:
    data = src.read()
    transform = src.transform
    raster_crs = src.crs
    rows, cols = src.height, src.width

if gdf.crs != raster_crs:
    gdf = gdf.to_crs(raster_crs)

if "Class" not in gdf.columns:
    # Support the class_name field used by the original notebook.
    if "class_name" in gdf.columns:
        gdf["Class"] = gdf["class_name"]
    else:
        raise KeyError("Sample file must contain 'Class' or 'class_name'.")

records = []

for idx, row in gdf.iterrows():
    x, y = row.geometry.x, row.geometry.y
    col_f, row_f = ~transform * (x, y)
    r, c = int(row_f), int(col_f)

    if not (0 <= r < rows and 0 <= c < cols):
        continue

    spectrum = data[:, r, c].astype(float)
    spectrum = spectrum[valid_band_mask]

    # Skip spectra with no usable signal.
    if not np.any(np.isfinite(spectrum)) or np.all(spectrum <= 0):
        continue

    record = {
        "Point_Index": idx,
        "Class": row["Class"],
        "X": x,
        "Y": y,
    }
    record.update({
        f"{wl:.2f}": value
        for wl, value in zip(clean_wavelengths, spectrum)
    })
    records.append(record)

spectra_df = pd.DataFrame(records)
spectra_df.to_csv(OUT_CSV, index=False)

print("Extracted spectra:", len(spectra_df))
print(spectra_df["Class"].value_counts())
print("Saved:", OUT_CSV.resolve())